# RNA Sequence Generation with RNAVerse

This notebook demonstrates how to generate RNA sequences using the RNAVerse model. RNAVerse supports three generation modes:

- **CLM (Causal Language Model)**: Autoregressive generation of complete RNA sequences, with optional conditioning on RNA type and/or species.
- **CLM Continuation**: Given a partial RNA sequence, extend it in the forward (5'→3') or reverse (3'→5') direction.
- **GLM (General Language Model)**: Span infilling — mask a region in an existing sequence and let the model regenerate it.

> Recommended runtime: use the `eva:latest` container.
> Inside the container, use `/composer-python/python` as the Python interpreter.
> The container does not include Jupyter by default, so install it first, for example with `/composer-python/python -m pip install jupyter nbconvert ipykernel`.

## Setup and Dependencies

Make sure you have the following packages installed:
```
torch >= 2.0
tokenizers >= 0.13
```

The `utils` toolkit (included in this repository under `tools/utils/`) handles model loading, prompt construction, and sequence generation.

In [ ]:
import sys

import torch

try:
    import generation_paths as GP
except ModuleNotFoundError:
    from notebooks.generation import generation_paths as GP

PROJECT_ROOT = GP.project_root()
TOOLS_DIR = GP.tools_dir()
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from utils.model.loader import ModelLoader
from utils.model.sampler import Sampler
from utils.generators.clm import CLMGenerator
from utils.generators.glm import GLMGenerator
from utils.conditions.condition import GenerationCondition
from utils.conditions.lineage import LineageDatabase
from utils.conditions.rna_types import list_rna_types

# Set random seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Model Initialization

Load the RNAVerse model from a checkpoint directory. The checkpoint should contain:
- `config.json` — model architecture configuration
- `tokenizer.json` — tokenizer vocabulary
- `model_weights.pt` (or any `.pt` file) — model weights

In [ ]:
CHECKPOINT_PATH = str(GP.checkpoint_dir("glm"))

loader = ModelLoader(checkpoint_path=CHECKPOINT_PATH)
model, tokenizer = loader.load(device="cuda:0")

print(f"Model loaded successfully.")
print(f"Model config: {loader.get_config()}")

## Explore Available Conditions

Before generating, let's see what RNA types and species are available for conditional generation.

In [ ]:
# List all supported RNA types
print("Supported RNA types:")
for rna_type in list_rna_types():
    print(f"  - {rna_type}")

print()

# List built-in species
lineage_db = LineageDatabase()
print("Built-in species:")
for sp in lineage_db.list_species():
    print(f"  - {sp['species_name']} (TaxID: {sp['taxid']})")

## 1. CLM Unconditional Generation

The simplest mode: generate RNA sequences from scratch with no constraints. The model starts from a `<bos>5` prompt and autoregressively produces nucleotides until it emits an end token.

Key parameters:
- `num_seqs`: number of sequences to generate
- `max_length`: maximum sequence length (keep it small for quick demos)
- `temperature`, `top_k`, `top_p`: sampling hyperparameters

In [ ]:
# Create a sampler and generator
sampler = Sampler(temperature=1.0, top_k=50)
clm_generator = CLMGenerator(model, tokenizer, sampler, device="cuda:0")

# Generate 3 unconditional RNA sequences
unconditional_seqs = clm_generator.generate(
    num_seqs=3,
    max_length=200,
)

print(f"Generated {len(unconditional_seqs)} sequences:\n")
for i, seq in enumerate(unconditional_seqs):
    print(f"Sequence {i+1} (length={len(seq)}):")
    print(f"  {seq[:80]}{'...' if len(seq) > 80 else ''}")

Let's inspect the prompt that was used under the hood:

In [ ]:
# The unconditional CLM prompt is simply:
prompt = clm_generator.build_prompt(condition=None)
print(f"Unconditional prompt: '{prompt}'")
# Expected: <bos>5

## 2. CLM Conditional Generation

RNAVerse can condition generation on **RNA type**, **species**, or **both**. The condition is encoded as a special prefix in the prompt.

Prompt format examples:
| Condition | Prompt |
|-----------|--------|
| RNA type only | `<bos>\|<rna_mRNA>\|5` |
| Species only | `<bos>\|d__eukaryota;...;s__homo_sapiens\|5` |
| Both | `<bos>\|d__eukaryota;...;s__homo_sapiens;<rna_mRNA>\|5` |

### 2.1 Condition on RNA type

In [ ]:
# Generate mRNA sequences
condition_rna = GenerationCondition(rna_type="mRNA")

# Inspect the prompt
print(f"Prompt: '{clm_generator.build_prompt(condition_rna)}'")

mrna_seqs = clm_generator.generate(
    condition=condition_rna,
    num_seqs=3,
    max_length=200,
)

print(f"\nGenerated {len(mrna_seqs)} mRNA sequences:")
for i, seq in enumerate(mrna_seqs):
    print(f"  [{i+1}] len={len(seq)}  {seq[:80]}...")

### 2.2 Condition on species

In [ ]:
# Generate human RNA sequences (TaxID 9606 = Homo sapiens)
condition_species = GenerationCondition(taxid="9606")

print(f"Prompt: '{clm_generator.build_prompt(condition_species)}'")

human_seqs = clm_generator.generate(
    condition=condition_species,
    num_seqs=3,
    max_length=200,
)

print(f"\nGenerated {len(human_seqs)} human RNA sequences:")
for i, seq in enumerate(human_seqs):
    print(f"  [{i+1}] len={len(seq)}  {seq[:80]}...")

### 2.3 Combined condition (RNA type + species)

In [ ]:
# Generate human tRNA sequences
condition_combined = GenerationCondition(rna_type="tRNA", taxid="9606")

print(f"Prompt: '{clm_generator.build_prompt(condition_combined)}'")

human_trna_seqs = clm_generator.generate(
    condition=condition_combined,
    num_seqs=3,
    max_length=200,
)

print(f"\nGenerated {len(human_trna_seqs)} human tRNA sequences:")
for i, seq in enumerate(human_trna_seqs):
    print(f"  [{i+1}] len={len(seq)}  {seq}")
    # tRNA sequences are typically 70-90 nt, so we print the full sequence

## 3. CLM Continuation (Sequence Extension)

Given an existing RNA sequence, the model can extend it in either direction:

- **Forward** (5'→3'): Keep the 5' end as prompt, generate the 3' end.
- **Reverse** (3'→5'): Keep the 3' end as prompt, generate the 5' end.

For reverse mode, the prompt is physically reversed before feeding to the model (since the model was trained on reversed sequences for the 3'→5' direction), and the output is reversed back.

You specify where to split the sequence using `split_pos` (absolute position) or `split_ratio` (fraction of sequence length).

### 3.1 Forward continuation (generate 3' end)

In [ ]:
# Example input sequence
example_seq = (
    "AUGCUAGCUAGCUAGCUAGCUAGCUAGCUAGCUAGCUAGCUAGCUAGCUAGC"
    "UAGCUAGCUAGCUAGCUAGCUAGCUAGCUAGCUAGCUAGCUAGCUAGCUAGC"
)
input_sequences = [("example_rna", example_seq)]

# Forward continuation: keep first 50% as prompt, generate the rest
forward_results = clm_generator.generate(
    input_sequences=input_sequences,
    direction="forward",
    split_ratio=0.5,
    num_seqs=2,       # generate 2 samples per input
    max_length=200,
)

print(f"Forward continuation results ({len(forward_results)} samples):\n")
for r in forward_results:
    print(f"Sample {r['sample_id']}:")
    print(f"  Prompt (5' end):       {r['prompt'][:50]}...")
    print(f"  Ground truth (3' end): {r['ground_truth'][:50]}...")
    print(f"  Generated (3' end):    {r['generated'][:50]}...")
    print(f"  Full sequence length:  {len(r['full'])}")
    print()

### 3.2 Reverse continuation (generate 5' end)

In [ ]:
# Reverse continuation: keep last 50% as prompt, generate the 5' end
reverse_results = clm_generator.generate(
    input_sequences=input_sequences,
    direction="reverse",
    split_ratio=0.5,
    num_seqs=2,
    max_length=200,
)

print(f"Reverse continuation results ({len(reverse_results)} samples):\n")
for r in reverse_results:
    print(f"Sample {r['sample_id']}:")
    print(f"  Prompt (3' end):       ...{r['prompt'][-50:]}")
    print(f"  Ground truth (5' end): {r['ground_truth'][:50]}...")
    print(f"  Generated (5' end):    {r['generated'][:50]}...")
    print(f"  Full sequence length:  {len(r['full'])}")
    print()

### 3.3 Continuation with conditions

You can combine continuation with conditional generation. For example, extend a viral RNA sequence while conditioning on the viral RNA type:

In [ ]:
condition_viral = GenerationCondition(rna_type="viral_RNA")

conditional_continuation = clm_generator.generate(
    condition=condition_viral,
    input_sequences=input_sequences,
    direction="forward",
    split_ratio=0.5,
    num_seqs=1,
    max_length=200,
)

r = conditional_continuation[0]
print(f"Conditional forward continuation (viral_RNA):")
print(f"  Generated 3' end: {r['generated'][:80]}...")
print(f"  Full length: {len(r['full'])}")

## 4. GLM Span Infilling

GLM mode masks a contiguous span within an existing sequence and asks the model to regenerate it. This is useful for:
- Sequence redesign (mutating a specific region)
- Evaluating model understanding of sequence context

The prompt format is:
```
<bos_glm>|<condition>|5[prefix]<span_X>[suffix]3<eos><span_X>
```
where `<span_X>` marks the masked region. The model generates new content for the masked span.

You control the span with:
- `span_length` (fixed number of nucleotides) or `span_ratio` (fraction of sequence)
- `span_position`: `"random"` or a specific start index

In [ ]:
# Create a GLM generator (reuses the same model and sampler)
glm_generator = GLMGenerator(model, tokenizer, sampler, device="cuda:0")

# Use the same example sequence
glm_input = [("example_rna", example_seq)]

# Mask 10% of the sequence at a random position
glm_results = glm_generator.generate(
    condition=GenerationCondition(rna_type="mRNA"),
    input_sequences=glm_input,
    span_ratio=0.1,
    span_position="random",
    num_seqs=2,
    max_length=200,
)

print(f"GLM Span Infilling results ({len(glm_results)} samples):\n")
for r in glm_results:
    print(f"Sample {r['sample_id']}:")
    print(f"  Span position: {r['span_start']}, Span length: {r['span_length']}")
    print(f"  Original span (ground truth): {r['ground_truth']}")
    print(f"  Generated span:               {r['generated']}")
    print(f"  Full reconstructed seq length: {len(r['full'])}")
    print()

### GLM with fixed span length and position

For more precise control, you can specify the exact span length and starting position:

In [ ]:
# Mask exactly 20 nucleotides starting at position 10
glm_fixed = glm_generator.generate(
    input_sequences=glm_input,
    span_length=20,
    span_position="10",
    num_seqs=1,
    max_length=200,
)

r = glm_fixed[0]
print(f"Fixed span infilling:")
print(f"  Prefix:       {r['prefix']}")
print(f"  Ground truth: {r['ground_truth']}")
print(f"  Generated:    {r['generated']}")
print(f"  Suffix:       {r['suffix'][:50]}...")

## 5. Sampling Parameters

The `Sampler` controls the randomness of generation. Let's compare different settings:

| Parameter | Effect |
|-----------|--------|
| `temperature` | < 1.0 = more deterministic, > 1.0 = more random |
| `top_k` | Only sample from the top K most probable tokens |
| `top_p` | Nucleus sampling — sample from the smallest set of tokens whose cumulative probability exceeds P |

In [ ]:
# Compare different temperature settings
condition = GenerationCondition(rna_type="mRNA", taxid="9606")

for temp in [0.5, 1.0, 1.5]:
    s = Sampler(temperature=temp, top_k=50)
    gen = CLMGenerator(model, tokenizer, s, device="cuda:0")
    seqs = gen.generate(condition=condition, num_seqs=1, max_length=100)
    print(f"temperature={temp}  len={len(seqs[0])}  {seqs[0][:60]}...")

In [ ]:
# Compare top_k values
for k in [4, 50, 200]:
    s = Sampler(temperature=1.0, top_k=k)
    gen = CLMGenerator(model, tokenizer, s, device="cuda:0")
    seqs = gen.generate(condition=condition, num_seqs=1, max_length=100)
    print(f"top_k={k:<4}  len={len(seqs[0])}  {seqs[0][:60]}...")

In [ ]:
# Compare top_p (nucleus) sampling
for p in [0.8, 0.95, 1.0]:
    s = Sampler(temperature=1.0, top_p=p)
    gen = CLMGenerator(model, tokenizer, s, device="cuda:0")
    seqs = gen.generate(condition=condition, num_seqs=1, max_length=100)
    print(f"top_p={p}  len={len(seqs[0])}  {seqs[0][:60]}...")

## 6. Saving Results to FASTA

Use the built-in FASTA writer to save generated sequences:

In [ ]:
from utils.io.fasta import write_fasta

# Prepare sequences as (header, sequence) tuples
fasta_records = [
    (f"generated_mRNA_{i+1}_len{len(seq)}", seq)
    for i, seq in enumerate(mrna_seqs)
]

output_path = str(GP.output_path("generated_mrna.fa"))
write_fasta(output_path, fasta_records)
print(f"Saved {len(fasta_records)} sequences to {output_path}")

# Preview the file
with open(output_path) as f:
    print(f.read()[:500])

## Summary

This notebook covered the core generation capabilities of RNAVerse:

| Mode | Use case | Key API |
|------|----------|---------|
| CLM unconditional | Generate diverse RNA from scratch | `CLMGenerator.generate()` |
| CLM conditional | Generate RNA of a specific type/species | `CLMGenerator.generate(condition=...)` |
| CLM continuation | Extend an existing sequence in either direction | `CLMGenerator.generate(input_sequences=..., direction=...)` |
| GLM span infilling | Redesign a region within a sequence | `GLMGenerator.generate(input_sequences=..., span_ratio=...)` |

For batch generation and multi-GPU parallel execution, see the command-line tool `generate.py` and the configuration guide in `README.md`.